In [1]:
#importing 
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset as HFDataset
import evaluate
import pandas as pd
import numpy as np
import torch

C:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Load processed data ───────────────────────────────────────
syn_train = pd.read_csv("../data/processed/syn_train.csv")
syn_val   = pd.read_csv("../data/processed/syn_val.csv")
syn_test  = pd.read_csv("../data/processed/syn_test.csv")

In [3]:
#Model selection
# MODEL_NAME = "xlm-roberta-base"
MODEL_NAME = "bert-base-multilingual-cased"


In [4]:
# ── Detect GPU & set memory-efficient dtype ────────────────────
device    = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16  = torch.cuda.is_available()   # auto-enable on GPU
print(f"Using device : {device}")
print(f"FP16 enabled : {use_fp16}")
print(f"Model        : {MODEL_NAME}\n")

Using device : cuda
FP16 enabled : True
Model        : bert-base-multilingual-cased



In [5]:
# ── Tokenizer ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["review_text"],
        truncation=True,
        padding="max_length",
        max_length=128          # 128 saves memory vs 512; increase if needed
    )

C:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [6]:
# ── Dataset helper ─────────────────────────────────────────────
def to_hf_dataset(df):
    return HFDataset.from_dict({
        "review_text": df["review_text"].tolist(),
        "label":       df["label"].tolist(),
        "word_count":  df["word_count"].tolist(),
        "rating":      df["rating"].tolist(),
        "slang_count": df["slang_count"].tolist(),
    })

train_hf = to_hf_dataset(syn_train).map(tokenize, batched=True)
val_hf   = to_hf_dataset(syn_val).map(tokenize,   batched=True)
test_hf  = to_hf_dataset(syn_test).map(tokenize,  batched=True)

Map: 100%|██████████████████████████████████████████████████████████████████| 448/448 [00:00<00:00, 7128.95 examples/s]


In [7]:
# ── Model ──────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

# ── Metrics ────────────────────────────────────────────────────
f1_metric  = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds          = np.argmax(logits, axis=1)
    return {
        "f1":       f1_metric.compute(
                        predictions=preds, references=labels,
                        average="weighted")["f1"],
        "accuracy": acc_metric.compute(
                        predictions=preds, references=labels)["accuracy"],
    }

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3972.67it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newl

In [8]:
# ── Early stopping callback ────────────────────────────────────
# Stops training if F1 doesn't improve for 3 consecutive eval epochs
early_stop = EarlyStoppingCallback(
    early_stopping_patience  = 3,
    early_stopping_threshold = 0.001   # minimum delta to count as improvement
)

In [9]:
# ── PRETRAIN on synthetic data ─────────────────────────────────
pretrain_args = TrainingArguments(
    output_dir              = "./pretrain_mbert_checkpoints",

    # ── Epochs & batch size ───────────────────────────────────
    num_train_epochs            = 10,
    per_device_train_batch_size = 16,   # reduce to 8 if OOM
    per_device_eval_batch_size  = 32,

    # ── Optimizer & scheduler ─────────────────────────────────
    learning_rate    = 2e-5,
    weight_decay     = 0.01,
    warmup_ratio     = 0.1,             # 10% of steps for warmup
    lr_scheduler_type= "cosine",        # cosine decay → better F1 than linear

    # ── Memory optimizations ──────────────────────────────────
    fp16                        = use_fp16,         # half-precision on GPU
    gradient_accumulation_steps = 2,                # effective batch = 16×2=32
    gradient_checkpointing      = True,             # trades compute for memory
    dataloader_pin_memory       = True,             # faster GPU data transfer
    dataloader_num_workers      = 4,                # parallel data loading

    # ── Evaluation & checkpointing ────────────────────────────
    eval_strategy     = "epoch",
    save_strategy           = "epoch",
    save_total_limit        = 3,        # keep only 3 best checkpoints on disk
    load_best_model_at_end  = True,
    metric_for_best_model   = "f1",
    greater_is_better       = True,

    # ── Logging ───────────────────────────────────────────────
    logging_dir             = "./logs",
    logging_steps           = 50,
    report_to               = "none",   # change to "wandb" if you use W&B
)

pretrain_trainer = Trainer(
    model           = model,
    args            = pretrain_args,
    train_dataset   = train_hf,
    eval_dataset    = val_hf,
    compute_metrics = compute_metrics,
    callbacks       = [early_stop],     # ← early stopping wired in
)

print("Starting pretraining on synthetic data...")
pretrain_trainer.train()
pretrain_trainer.save_model("../models/transformers/pretrained_mbert_model")
tokenizer.save_pretrained("../models/transformers/pretrained_mbert_model")  # save tokenizer alongside model
print("Pretraining complete. Model saved to ../models/transformers/pretrained_mbert_model\n")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting pretraining on synthetic data...


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.378267,0.107265,0.970920,0.970917
2,0.134814,0.057637,0.986579,0.986577
3,0.083883,0.061203,0.984339,0.984340
4,0.044115,0.035772,0.993287,0.993289
5,0.015559,0.019365,0.995526,0.995526
6,0.000701,0.022013,0.995526,0.995526
7,0.002804,0.035705,0.995526,0.995526
8,0.001034,0.037048,0.995526,0.995526


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.14s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.l

Pretraining complete. Model saved to ../models/transformers/pretrained_mbert_model

